# Premier League Standings ETL Pipeline
**Flow:** API-Football → raw JSON → pandas DataFrame → staging tables → stored procedures (UPSERT) → warehouse tables in SQL Server → Power BI

| Stage | What happens | Where |
|---|---|---|
| Extract | GET request to the API, reply saved as raw JSON | Cells 3 |
| Transform | Parse JSON, flatten into clean DataFrames | Cells 4-6 |
| Load | Build warehouse in SQL, upsert via staging | Cells 7-12 |
| Check | Verify counts, duplicates, joins, re-run safety | Cells 13-16 |
| Visualize | Power BI reads the warehouse tables | Power BI |

**Warehouse design:** `dim_team` (who the team is) + `pl_standings` (the numbers), linked by `team_id`.

## 1. Setup
Install the libraries once, import them, and keep **every setting in one config cell**. The API key is read from a `.env` file, so no secret appears in the code or on screen.

In [1]:
#Cell-1
%pip install requests pandas sqlalchemy pyodbc python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Cell-2
import os
import json
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()   # reads API_KEY and BASE_URL from .env

# ---- API settings ----
# Key and URL come from .env, so the secret never appears in the notebook
# LEAGUE / SEASON are variables: change one line to load another season

API_KEY  = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
LEAGUE   = 39      # Premier League
SEASON   = 2024

# ---- SQL Server settings ----
# Blank user/password = Windows Authentication (no password stored anywhere)

server   = os.getenv("MSSQL_SERVER", r".\SQLEXPRESS")  # local default instance; set MSSQL_SERVER in .env
database = "PremierLeagueDB"
user     = ""      # blank = Windows Authentication
password = ""      # blank = Windows Authentication
driver   = "ODBC Driver 17 for SQL Server"

# Safe check: never prints the key itself
print("API_KEY loaded:", API_KEY is not None)
print("BASE_URL:", BASE_URL)
print("League / Season:", LEAGUE, "/", SEASON)
print("SQL Server:", server, "| Database:", database)

API_KEY loaded: True
BASE_URL: https://v3.football.api-sports.io
League / Season: 39 / 2024
SQL Server: .\SQLEXPRESS | Database: PremierLeagueDB


## 2. Extract
Ask the API for data and **save the reply exactly as received** (raw JSON). One small function sends a GET request, and we call it for two endpoints: `/standings` (the numbers) and `/teams` (team and stadium details).
- URL = where to ask, headers = who I am (API key), params = what I want (league 39, season 2024)
- HTTP 200 only means the server answered. The real check for errors comes in the next section.

In [3]:
#Cell-3
headers = {"x-apisports-key": API_KEY}             # identifies me to the API
params  = {"league": LEAGUE, "season": SEASON}     # what data I want

def extract(endpoint):
    """Send a GET request to one endpoint and return the raw reply text."""
    response = requests.get(f"{BASE_URL}/{endpoint}", headers=headers, params=params)
    print(f"/{endpoint}  HTTP status:", response.status_code,
          "| characters:", len(response.text))
    return response.text

standings_raw_json = extract("standings")         # raw reply, untouched
teams_raw_json     = extract("teams")             # raw reply, untouched

print(standings_raw_json[:200])
print(teams_raw_json[:200])

/standings  HTTP status: 200 | characters: 10328
/teams  HTTP status: 200 | characters: 7540
{"get":"standings","parameters":{"league":"39","season":"2024"},"errors":[],"results":1,"paging":{"current":1,"total":1},"response":[{"league":{"id":39,"name":"Premier League","country":"England","log
{"get":"teams","parameters":{"league":"39","season":"2024"},"errors":[],"results":20,"paging":{"current":1,"total":1},"response":[{"team":{"id":33,"name":"Manchester United","code":"MUN","country":"En


## 3. Transform
Raw JSON is just text. Three moves turn it into something usable:
1. `json.loads()` converts text into Python dictionaries and lists
2. **Check the `errors` field.** The API can return HTTP 200 and still report a problem, so we stop here if it did
3. Pick out only the part that holds the data. For standings: `response[0]['league']['standings'][0]`, a list with one dictionary per team

In [4]:
#Cell-4
# Text -> Python objects
# Stop if the API reported errors, even when HTTP status was 200
# Standings data is nested: response -> league -> standings -> first table (list of 20 team dicts)
# Teams data is simpler: the "response" list itself holds the 20 teams

standings_data = json.loads(standings_raw_json)
teams_data     = json.loads(teams_raw_json)

# HTTP 200 can still carry errors, so check both replies
for name, d in [("standings", standings_data), ("teams", teams_data)]:
    if d["errors"]:
        raise Exception(f"{name} returned errors: {d['errors']}")

standings_list = standings_data["response"][0]["league"]["standings"][0]
teams_list     = teams_data["response"]

print("Standings rows:", len(standings_list))
print("Teams rows:", len(teams_list))

Standings rows: 20
Teams rows: 20


**Flatten into DataFrames.** Nested fields like `team.id` or `all.goals.for` become normal columns with `pd.json_normalize()`. We keep only the columns we need and rename them to clean SQL-friendly names.
- `season` is added by us, because it completes the **composite key (season + team_id)**
- Both tables share `team_id`. That is the link between them in the warehouse.

In [5]:
# Flatten nested JSON into columns (team.id, all.goals.for ...)
# Keep only the columns we need
# Rename to SQL-friendly names
# Add season: the API doesn't put it in each row, but our key needs it
# isna().sum() = count of missing values per column (NULLs allowed in SQL for these)

In [6]:
#Cell-5
df_standings = pd.json_normalize(standings_list)[[
    "rank", "team.id", "team.name", "points", "goalsDiff", "form",
    "all.played", "all.win", "all.draw", "all.lose",
    "all.goals.for", "all.goals.against"
]].copy()

df_standings.columns = [
    "rank", "team_id", "team_name", "points", "goals_diff", "form",
    "played", "won", "drawn", "lost", "goals_for", "goals_against"
]

df_standings.insert(0, "season", SEASON)   # part of the composite key

print(df_standings.shape)
print(df_standings.head(3))

(20, 13)
   season  rank  team_id        team_name  points  goals_diff   form  played  \
0    2024     1       40        Liverpool      84          45  DLDLW      38   
1    2024     2       42          Arsenal      74          35  WWDLD      38   
2    2024     3       50  Manchester City      71          28  WWDWW      38   

   won  drawn  lost  goals_for  goals_against  
0   25      9     4         86             41  
1   20     14     4         69             34  
2   21      8     9         72             44  


In [7]:
#Cell-6
df_teams = pd.json_normalize(teams_list)[[
    "team.id", "team.name", "team.code", "team.country", "team.founded",
    "venue.name", "venue.city", "venue.capacity"
]].copy()

df_teams.columns = [
    "team_id", "team_name", "team_code", "country", "founded",
    "venue_name", "venue_city", "venue_capacity"
]

print(df_teams.shape)
print(df_teams.head(3))
print(df_teams.isna().sum())   # count of missing values per column

(20, 8)
   team_id          team_name team_code  country  founded        venue_name  \
0       33  Manchester United       MUN  England     1878      Old Trafford   
1       34          Newcastle       NEW  England     1892   St. James' Park   
2       35        Bournemouth       BOU  England     1899  Vitality Stadium   

            venue_city  venue_capacity  
0           Manchester           76212  
1  Newcastle upon Tyne           52758  
2  Bournemouth, Dorset           12000  
team_id           0
team_name         0
team_code         0
country           0
founded           0
venue_name        0
venue_city        0
venue_capacity    0
dtype: int64


## 4. Build the warehouse in SQL
Python only carries the SQL to SQL Server. The SQL itself sits in readable blocks, in this order:
1. Rebuild a clean `PremierLeagueDB`
2. Create the warehouse tables, staging tables and foreign key
3. Create two stored procedures that do the UPSERT
4. List what now exists

In [8]:
#Cell-7
# Connect to "master" because PremierLeagueDB doesn't exist yet
# AUTOCOMMIT: SQL Server doesn't allow CREATE/DROP DATABASE inside a transaction
# WARNING: this cell deletes the database and rebuilds it. Re-running wipes everything.

master_url = (f"mssql+pyodbc://@{server}/master"
              f"?driver={driver.replace(' ', '+')}&trusted_connection=yes")
master_engine = create_engine(master_url, isolation_level="AUTOCOMMIT")

drop_db_sql = """
IF DB_ID('PremierLeagueDB') IS NOT NULL
BEGIN
    ALTER DATABASE PremierLeagueDB SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE PremierLeagueDB;
END
"""
create_db_sql = "CREATE DATABASE PremierLeagueDB"

with master_engine.connect() as conn:
    conn.execute(text(drop_db_sql))
    conn.execute(text(create_db_sql))
    print("Database created:", conn.execute(text(
        "SELECT name FROM sys.databases WHERE name = 'PremierLeagueDB'")).scalar())

Database created: PremierLeagueDB


In [9]:
#Cell-8
# Second engine, now pointing at PremierLeagueDB
# trusted_connection=yes = Windows Authentication

db_url = (f"mssql+pyodbc://@{server}/{database}"
          f"?driver={driver.replace(' ', '+')}&trusted_connection=yes")
engine = create_engine(db_url)

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT DB_NAME()")).scalar())

Connected to: PremierLeagueDB


### Building the warehouse tables (Cell 9)
This cell creates four tables in `PremierLeagueDB`, in two groups.

**Warehouse tables (the final data)**

| Table | Role | Key |
|---|---|---|
| `dim_team` | Dimension: who the team is (name, stadium, city, capacity) | `team_id` |
| `pl_standings` | Fact: the numbers (points, wins, goals) | `season` + `team_id` |

**Staging tables (temporary landing area)**
`stg_dim_team` and `stg_pl_standings` have the same columns but no keys. They hold only the current batch before the stored procedures move it into the warehouse.

**Two rules the database enforces for us:**
- **Composite primary key (season + team_id):** the same team can't appear twice in one season, so duplicates are impossible
- **Foreign key (`team_id`):** a standings row must point to a real team in `dim_team`, which is why teams are loaded before standings

All tables are empty after this cell. The data arrives in the Load step.

In [10]:
#Cell-9
create_tables_sql = """
-- ======================================================
-- DIMENSION TABLE: dim_team
-- Describes WHO the team is. One row per team, regardless of season.
-- ======================================================
CREATE TABLE dbo.dim_team (
    team_id        INT            NOT NULL PRIMARY KEY,  -- unique id from the API; the link to standings
    team_name      NVARCHAR(100)  NOT NULL,
    team_code      NVARCHAR(10)   NULL,                  -- NULL allowed: API may not supply every detail
    country        NVARCHAR(50)   NULL,
    founded        INT            NULL,
    venue_name     NVARCHAR(150)  NULL,
    venue_city     NVARCHAR(100)  NULL,
    venue_capacity INT            NULL
);

-- ======================================================
-- STANDINGS TABLE: pl_standings
-- Holds the NUMBERS. One row per team per season.
-- ======================================================
CREATE TABLE dbo.pl_standings (
    season         INT            NOT NULL,
    team_id        INT            NOT NULL,
    rank           INT            NOT NULL,
    team_name      NVARCHAR(100)  NOT NULL,
    points         INT            NOT NULL,
    goals_diff     INT            NOT NULL,
    form           NVARCHAR(20)   NULL,                  -- can be empty early in a season
    played         INT            NOT NULL,
    won            INT            NOT NULL,
    drawn          INT            NOT NULL,
    lost           INT            NOT NULL,
    goals_for      INT            NOT NULL,
    goals_against  INT            NOT NULL,

    -- COMPOSITE PRIMARY KEY: a team can appear only once per season,
    -- so the database itself blocks duplicates
    CONSTRAINT PK_pl_standings PRIMARY KEY (season, team_id),

    -- FOREIGN KEY: every standings row must point to a real team in dim_team
    -- (this is why teams must be loaded BEFORE standings)
    CONSTRAINT FK_standings_team FOREIGN KEY (team_id) REFERENCES dbo.dim_team (team_id)
);

-- ======================================================
-- STAGING TABLES: temporary landing area for each new batch
-- Same columns as the warehouse tables, but no keys.
-- WHERE 1 = 0 copies the structure only, with no rows.
-- ======================================================
SELECT * INTO dbo.stg_dim_team     FROM dbo.dim_team     WHERE 1 = 0;
SELECT * INTO dbo.stg_pl_standings FROM dbo.pl_standings WHERE 1 = 0;
"""

# Python only carries the SQL to the server; engine.begin() runs it as one transaction
with engine.begin() as conn:
    conn.execute(text(create_tables_sql))
print("Tables created")

Tables created


### How the UPSERT works (read this before running Cell 10)
A stored procedure is a saved block of SQL that lives inside the database. Ours does two things, inside one transaction:

| Step | Statement | What it does |
|---|---|---|
| 1 | `UPDATE ... JOIN` | Rows already in the warehouse get fresh values from staging |
| 2 | `INSERT ... WHERE NOT EXISTS` | Only rows not yet in the warehouse are added |

**Why this matters:** run the notebook twice, or after a match week, and we get updated numbers with **no duplicates**. If anything fails halfway, the transaction rolls back and the table is never left half-loaded.

In [11]:
#Cell-10
# ======================================================
# PROCEDURE 1: upsert the team dimension
# UPSERT = UPDATE rows that exist + INSERT rows that are new
# ======================================================
upsert_team_sql = """
CREATE OR ALTER PROCEDURE dbo.usp_upsert_dim_team   -- CREATE OR ALTER: safe to re-run
AS
BEGIN
    SET NOCOUNT ON;       -- hide row-count messages
    SET XACT_ABORT ON;    -- on any error, roll back automatically
    BEGIN TRANSACTION;    -- update + insert succeed together or not at all

    -- 1) UPDATE teams that already exist in the warehouse
    --    (picks up changes such as a renamed stadium or new capacity)
    UPDATE t
    SET t.team_name = s.team_name, t.team_code = s.team_code,
        t.country = s.country, t.founded = s.founded,
        t.venue_name = s.venue_name, t.venue_city = s.venue_city,
        t.venue_capacity = s.venue_capacity
    FROM dbo.dim_team AS t
    JOIN dbo.stg_dim_team AS s ON t.team_id = s.team_id;   -- match on team_id

    -- 2) INSERT teams that are new (not yet in the warehouse)
    INSERT INTO dbo.dim_team
        (team_id, team_name, team_code, country, founded,
         venue_name, venue_city, venue_capacity)
    SELECT s.team_id, s.team_name, s.team_code, s.country, s.founded,
           s.venue_name, s.venue_city, s.venue_capacity
    FROM dbo.stg_dim_team AS s
    WHERE NOT EXISTS (SELECT 1 FROM dbo.dim_team AS t WHERE t.team_id = s.team_id);
    -- NOT EXISTS is what prevents duplicates: existing teams are skipped

    COMMIT TRANSACTION;
END
"""

# ======================================================
# PROCEDURE 2: upsert the standings
# Same pattern, but matched on the composite key (season + team_id)
# ======================================================
upsert_standings_sql = """
CREATE OR ALTER PROCEDURE dbo.usp_upsert_pl_standings
AS
BEGIN
    SET NOCOUNT ON;
    SET XACT_ABORT ON;
    BEGIN TRANSACTION;

    -- 1) UPDATE rows that already exist (same season + team_id)
    --    Points, rank and goals change every match week, so existing rows get refreshed
    UPDATE t
    SET t.rank = s.rank, t.team_name = s.team_name, t.points = s.points,
        t.goals_diff = s.goals_diff, t.form = s.form, t.played = s.played,
        t.won = s.won, t.drawn = s.drawn, t.lost = s.lost,
        t.goals_for = s.goals_for, t.goals_against = s.goals_against
    FROM dbo.pl_standings AS t
    JOIN dbo.stg_pl_standings AS s
      ON t.season = s.season AND t.team_id = s.team_id;

    -- 2) INSERT rows that are new (this season + team combination not in the table yet)
    INSERT INTO dbo.pl_standings
        (season, team_id, rank, team_name, points, goals_diff, form,
         played, won, drawn, lost, goals_for, goals_against)
    SELECT s.season, s.team_id, s.rank, s.team_name, s.points, s.goals_diff, s.form,
           s.played, s.won, s.drawn, s.lost, s.goals_for, s.goals_against
    FROM dbo.stg_pl_standings AS s
    WHERE NOT EXISTS (
        SELECT 1 FROM dbo.pl_standings AS t
        WHERE t.season = s.season AND t.team_id = s.team_id);

    COMMIT TRANSACTION;
END
"""

# Each CREATE PROCEDURE runs as its own statement
with engine.begin() as conn:
    conn.execute(text(upsert_team_sql))
    conn.execute(text(upsert_standings_sql))
print("Procedures created")

Procedures created


In [18]:
#Cell-11
# Ask SQL Server itself what exists: 4 tables (2 warehouse + 2 staging), 2 procedures

with engine.connect() as conn:
    tables = [r[0] for r in conn.execute(text("SELECT name FROM sys.tables ORDER BY name"))]
    procs  = [r[0] for r in conn.execute(text("SELECT name FROM sys.procedures ORDER BY name"))]
print("Tables:", tables)
print("Procedures:", procs)

Tables: ['dim_team', 'pl_standings', 'stg_dim_team', 'stg_pl_standings']
Procedures: ['usp_upsert_dim_team', 'usp_upsert_pl_standings']


## 5. Load
Each DataFrame goes into a **staging table**, then a **stored procedure** upserts it into the warehouse table.
- **Order matters:** teams first, because standings has a foreign key to `dim_team`
- **One transaction:** if any step fails, nothing is changed
- **TRUNCATE first:** staging must hold only the current batch

In [25]:
#Cell-12
with engine.begin() as conn:   # one transaction: commits only if every step works

    # 1) Teams: clear staging -> copy DataFrame in -> procedure upserts into dim_team
    conn.execute(text("TRUNCATE TABLE dbo.stg_dim_team"))
    df_teams.to_sql("stg_dim_team", conn, schema="dbo",
                    if_exists="append", index=False)     # append to existing table; no pandas index column
    conn.execute(text("EXEC dbo.usp_upsert_dim_team"))

    # 2) Standings: same pattern, after teams (foreign key dependency) - staging, then upsert into pl_standings
    conn.execute(text("TRUNCATE TABLE dbo.stg_pl_standings"))
    df_standings.to_sql("stg_pl_standings", conn, schema="dbo",
                        if_exists="append", index=False)   # append to existing table; no pandas index column
    conn.execute(text("EXEC dbo.usp_upsert_pl_standings"))

print("Load finished")

Load finished


## 6. Check the load
"Load finished" is not proof, so we ask SQL Server directly:
- **Row counts:** 20 teams, 20 standings rows
- **Duplicate check:** any season + team_id appearing more than once? Expected 0
- **Join:** standings linked to team details through `team_id`
- **Reload test:** run the load again. The counts stay at 20, so no duplicates are created.

In [26]:
#Cell-13
check_counts_sql = """                 
-- Row count per table (UNION ALL stacks the two results)
SELECT 'dim_team'     AS table_name, COUNT(*) AS row_count FROM dbo.dim_team    
UNION ALL
SELECT 'pl_standings' AS table_name, COUNT(*) AS row_count FROM dbo.pl_standings
"""

check_dupes_sql = """
-- Groups by the composite key; HAVING COUNT(*) > 1 would reveal any duplicate. Expect 0.
SELECT COUNT(*) AS duplicate_keys
FROM (
    SELECT season, team_id
    FROM dbo.pl_standings
    GROUP BY season, team_id
    HAVING COUNT(*) > 1
) AS d
"""

print(pd.read_sql(check_counts_sql, engine))
print(pd.read_sql(check_dupes_sql, engine))

     table_name  row_count
0      dim_team         20
1  pl_standings         20
   duplicate_keys
0               0


In [27]:
#Cell-14
join_sql = """
-- Join: numbers from pl_standings + stadium details from dim_team, linked by team_id
SELECT TOP 5
    s.season, s.rank, t.team_name, t.venue_name, t.venue_capacity,
    s.points, s.goals_diff
FROM dbo.pl_standings AS s
JOIN dbo.dim_team AS t ON s.team_id = t.team_id
ORDER BY s.rank
"""

print(pd.read_sql(join_sql, engine))

   season  rank        team_name        venue_name  venue_capacity  points  \
0    2024     1        Liverpool           Anfield           61276      84   
1    2024     2          Arsenal  Emirates Stadium           60383      74   
2    2024     3  Manchester City    Etihad Stadium           55097      71   
3    2024     4          Chelsea   Stamford Bridge           41841      69   
4    2024     5        Newcastle   St. James' Park           52758      66   

   goals_diff  
0          45  
1          35  
2          28  
3          21  
4          21  


In [30]:
#Cell-15
# Same load as Cell 12, run a second time: counts must still be 20 (upsert, not duplicate)

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE dbo.stg_dim_team"))
    df_teams.to_sql("stg_dim_team", conn, schema="dbo", if_exists="append", index=False)
    conn.execute(text("EXEC dbo.usp_upsert_dim_team"))

    conn.execute(text("TRUNCATE TABLE dbo.stg_pl_standings"))
    df_standings.to_sql("stg_pl_standings", conn, schema="dbo", if_exists="append", index=False)
    conn.execute(text("EXEC dbo.usp_upsert_pl_standings"))

print(pd.read_sql(check_counts_sql, engine))

     table_name  row_count
0      dim_team         20
1  pl_standings         20


## 7. Visualize (Power BI)
Power BI connects to SQL Server through Get Data and imports `dim_team` and `pl_standings`. I linked them on `team_id` (one team to many standings rows), so one visual can show stadium details next to league points. Staging tables are not imported because they are only landing areas.